In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle

In [4]:

np.random.seed(42)

# === PARAMETRY — zmień tutaj ===
N_NORMAL = 2000      # liczba normalnych transakcji
N_FRAUD  = 100       # liczba fraudów
# ===============================

# Normalne transakcje
normal = pd.DataFrame({
    'amount': np.random.lognormal(5, 1, N_NORMAL).clip(5, 5000),
    'is_electronics': np.random.binomial(1, 0.3, N_NORMAL),
    'tx_per_minute': np.random.poisson(3, N_NORMAL),
    'fraud': 0
})


# Fraudy
fraud = pd.DataFrame({
    'amount': np.random.uniform(2000, 9000, N_FRAUD),
    'is_electronics': np.random.binomial(1, 0.7, N_FRAUD),
    'tx_per_minute': np.random.poisson(8, N_FRAUD),
    'fraud': 1
})

df = pd.concat([normal, fraud], ignore_index=True).sample(frac=1, random_state=42)
print(f"Dataset: {len(df)} wierszy, fraud rate: {df['fraud'].mean():.1%}")

Dataset: 2100 wierszy, fraud rate: 4.8%


In [5]:
features = ['amount', 'is_electronics', 'tx_per_minute']
X = df[features]
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
)

# TWÓJ KOD
# 1. train_test_split (80/20, stratify=y)
# 2. RandomForestClassifier(100)
# 3. classification_report
# 4. pickle.dump do 'fraud_model.pkl'

In [6]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred=rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       400
           1       1.00      1.00      1.00        20

    accuracy                           1.00       420
   macro avg       1.00      1.00      1.00       420
weighted avg       1.00      1.00      1.00       420



In [8]:
with open('fraud_model.pkl', 'wb') as f:
    pickle.dump(rf, f)

In [9]:
%%file fraud_api.py
from fastapi import FastAPI
from pydantic import BaseModel
import pickle, numpy as np

app = FastAPI(title="Fraud Detection API")
model = pickle.load(open('fraud_model.pkl', 'rb'))

class Transaction(BaseModel):
    amount: float
    is_electronics: int
    tx_per_minute: int

@app.post("/score")
def score_transaction(tx: Transaction):
    features = np.array([[tx.amount, tx.is_electronics, tx.tx_per_minute]])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0][1]
    return {
        "is_fraud": bool(prediction),
        "fraud_probability": float(probability)
    }

@app.get("/health")
def health():
    return {"status": "ok"}


Overwriting fraud_api.py


In [2]:
import requests

# Test normalna
r = requests.post("http://localhost:8001/score",
    json={"amount": 150, "is_electronics": 0, "tx_per_minute": 3})
print("Normalna:", r.json())

# Test podejrzana
r = requests.post("http://localhost:8001/score",
    json={"amount": 5500, "is_electronics": 1, "tx_per_minute": 12})
print("Podejrzana:", r.json())


Normalna: {'is_fraud': False, 'fraud_probability': 0.0}
Podejrzana: {'is_fraud': True, 'fraud_probability': 0.99}


In [10]:
%%file ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
from datetime import datetime
import json, requests

consumer = KafkaConsumer('lab1', bootstrap_servers='broker:9092',
    auto_offset_reset='earliest', group_id='ml-scoring',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')))

alert_producer = KafkaProducer(bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8'))

API_URL = "http://localhost:8001/score"

for message in consumer:
    tx = message.value

    features = {
        "amount": tx["amount"],
        "is_electronics": 1 if tx["category"] == "elektronika" else 0,
        "tx_per_minute": 5
    }

    response = requests.post(API_URL, json=features)
    result = response.json()

    if result["is_fraud"]:
        alert = {
            "timestamp": datetime.now().isoformat(),
            "transaction": tx,
            "fraud_probability": result["fraud_probability"]
        }
        alert_producer.send('alerts', alert)
        print(f"🚨 ALERT: {tx['tx_id']} - fraud_prob={result['fraud_probability']:.2%}")

Overwriting ml_consumer.py
